# Naive Baseline

Evaluate a fixed-origin persistence forecast on the shared chronological test window.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "tcs_stock_data_cleaned.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_predictions.csv"


In [ ]:
stock_data = pd.read_csv(DATA_PATH, parse_dates=["Date"])
stock_data = stock_data.sort_values("Date").reset_index(drop=True)
split_index = int(len(stock_data) * 0.8)
train = stock_data.iloc[:split_index]
test = stock_data.iloc[split_index:].copy()


In [ ]:
forecast = np.repeat(float(train["Close"].iloc[-1]), len(test))
results = pd.DataFrame(
    {
        "Date": test["Date"],
        "Actual": test["Close"].to_numpy(),
        "Prediction": forecast,
    }
)


In [ ]:
mae = mean_absolute_error(results["Actual"], results["Prediction"])
rmse = np.sqrt(mean_squared_error(results["Actual"], results["Prediction"]))
nonzero = results["Actual"].ne(0)
mape = (
    (results.loc[nonzero, "Actual"] - results.loc[nonzero, "Prediction"]).abs()
    .div(results.loc[nonzero, "Actual"])
    .mean()
    * 100
)
pd.Series({"MAE": mae, "RMSE": rmse, "MAPE (%)": mape})


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(results["Date"], results["Actual"], label="Actual")
ax.plot(results["Date"], results["Prediction"], label="Fixed-origin baseline")
ax.set(title="Naive Baseline vs Actual", xlabel="Date", ylabel="Closing Price (INR)")
ax.legend()
fig.autofmt_xdate()
plt.show()


In [ ]:
results.to_csv(OUTPUT_PATH, index=False)
